<a href="https://colab.research.google.com/github/izzumani/artificial-engineering/blob/main/Using_word_embeddings_for_spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import argparse
import gensim.downloader as api
import numpy as np
import os
import shutil
import tensorflow as tf
from sklearn.metrics import accuracy_score,confusion_matrix


In [2]:
def download_and_read(url):
  local_file = url.split('/')[-1]
  p = tf.keras.utils.get_file(local_file, url, extract=True, cache_dir='.')
  labels, texts = [],[]
  with open("/content/datasets/SMSSpamCollection", "r") as fin:
    for line in fin:
      label,text = line.strip().split('\t')
      labels.append(1 if label == 'spam' else 0)
      texts.append(text)
  return texts, labels
DATASET_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
texts, labels = download_and_read(DATASET_URL)

 147456/Unknown 0s 1us/step

In [3]:
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
# print(sequences)
text_sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences)
# print(text_sequences)
num_records = len(text_sequences)
max_seqlen = len(text_sequences[0])
print("{:d} sentences, max length: {:d}".format(num_records, max_seqlen))

5574 sentences, max length: 189


In [4]:
print(labels)

[0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [5]:
NUM_CLASSES = 2
cat_labels  = tf.keras.utils.to_categorical(labels, num_classes=NUM_CLASSES)


In [6]:
word2idx = tokenizer.word_index

idx2word = {v:k for k, v in word2idx.items()}

word2idx["PAD"] = 0
idx2word[0] = "PAD"
vocab_size = len(word2idx)
print("vocab size: {:d}".format(vocab_size))

vocab size: 9010


In [7]:
dataset = tf.data.Dataset.from_tensor_slices((text_sequences, cat_labels))
dataset = dataset.shuffle(10000)
test_size = num_records // 4
val_size = (num_records - test_size) // 10
test_dataset = dataset.take(test_size)
val_dataset = dataset.skip(test_size).take(val_size)
train_dataset = dataset.skip(test_size + val_size)
BATCH_SIZE = 128
test_dataset = test_dataset.batch(BATCH_SIZE, drop_remainder=True)
val_dataset = val_dataset.batch(BATCH_SIZE, drop_remainder=True)
train_dataset = train_dataset.batch(BATCH_SIZE, drop_remainder=True)

In [8]:
import gensim.downloader as api
print(api.info()['models'].keys())

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])


In [9]:
EMBEDDING_DIM = 300
DATA_DIR = "/content/data/E.npy"
EMBEDDING_NUMPY_FILE = "/content/data/E.npy"
EMBEDDING_MODEL = "glove-wiki-gigaword-300"

In [10]:
def build_embedding_matrix(sequences, word2idx, embedding_dim, embedding_file):
  if os.path.exists(embedding_file):
    E = np.load(embedding_file)
  else:
    vocab_size = len(word2idx)
    E = np.zeros((vocab_size, embedding_dim))
    word_vectors = api.load(EMBEDDING_MODEL)
    for word, idx in word2idx.items():
      try:
        E[idx] = word_vectors.get_vector(word)
      except KeyError:
        pass
    np.save(embedding_file, E)
  return E

In [11]:
E = build_embedding_matrix(text_sequences, word2idx,EMBEDDING_DIM,EMBEDDING_NUMPY_FILE)
print("Embedding matrix:", E.shape)

[==================================================] 100.0% 376.1/376.1MB downloaded
Embedding matrix: (9010, 300)


<ipython-input-10-05cb07296369>:10: DeprecationWarning: Call to deprecated `word_vec` (Use get_vector instead).
  E[idx] = word_vectors.word_vec(word)


In [12]:
class SpamClassifier(tf.keras.Model):
  def __init__(self, vocab_sz, emb_sz, input_length, num_filters, kernel_sz, output_sz,run_mode, embedding_weigts, **kwargs):
    if run_mode =="scratch":
      self.embedding = tf.keras.layers.Embedding(vocab_sz, emb_sz, input_length=input_length,trainable=True)
    elif run_mode == "vectorizer":
      self.embedding = tf.keras.layers.Embedding(vocab_sz, emb_sz, input_length=input_length,weights=[embedding_weigts], trainable=False)
    else:
      self.embedding = tf.keras.layers.Embedding(vocab_sz, emb_sz, input_length=input_length,weights=[embedding_weigts], trainable=True)
    self.conv = tf.keras.layers.Conv1D(filters=num_filters, kernel_size=kernel_sz, activation="relu")
    self.dropout = tf.keras.layers.SpatialDropout1D(0.2)
    self.pool = tf.keras.layers.GlobalMaxPooling1D()
    self.dense = tf.keras.layers.Dense(output_sz, activation="softmax")
    super(SpamClassifier, self).__init__(**kwargs)
  def call(self,x):
    x = self.embedding(x)
    x = self.conv(x)
    x = self.dropout(x)
    x = self.pool(x)
    x = self.dense(x)
    return x


In [14]:
conv_num_filters = 256
conv_kernel_size = 3
model = SpamClassifier(vocab_size, EMBEDDING_DIM, max_seqlen, conv_num_filters, conv_kernel_size, NUM_CLASSES, "vectorizer", E)
model.build(input_shape=(None, max_seqlen))
model.summary()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:391: UserWarning: `build()` was called on layer 'spam_classifier_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "spam_classifier_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │       2,703,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d_1                  │ ?                           │     0 (unbuilt) │
│ (SpatialDropout1D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d_1               │ ?                           │     0 (unbuilt) │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,703,000 (10.31 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,703,000 (10.31 MB)

In [15]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])


In [16]:
NUM_EPOCHS = 3
CLASS_WEIGHTS = {0:1, 1:8}
model.fit(train_dataset, epochs=NUM_EPOCHS, validation_data=val_dataset, class_weight=CLASS_WEIGHTS)

Epoch 1/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.7989 - loss: 0.7349 - val_accuracy: 0.9583 - val_loss: 0.1329
Epoch 2/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9620 - loss: 0.2068 - val_accuracy: 0.9844 - val_loss: 0.0567
Epoch 3/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9881 - loss: 0.0985 - val_accuracy: 0.9974 - val_loss: 0.0353


In [17]:
labels, predictions = [], []
for Xtest, Ytest in test_dataset:
  Ytest_ = model.predict_on_batch(Xtest)
  ytest = np.argmax(Ytest, axis=1)
  ytest_ = np.argmax(Ytest_, axis=1)
  labels.extend(ytest.tolist())
  predictions.extend(ytest_.tolist())
print("accuracy: {:.3f}".format(accuracy_score(labels, predictions)))
print("confusion matrix")
confusion_matrix(labels, predictions)

accuracy: 0.994
confusion matrix


array([[1096,    8],
       [   0,  176]])